In [0]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

In [0]:
dbutils.widgets.get("job_parameters")

In [0]:
parameters = json.loads(dbutils.widgets.get("job_parameters"))
catalog = parameters.get("catalog")
source_view_name = catalog + "." + parameters.get("source_view")
target_table_name = catalog + "." + parameters.get("target_table")
primary_keys = parameters.get("primary_keys")
job_id = parameters.get("job_id")
parent_job = parameters.get("parent_job_id", job_id)
partition = parameters.get("partition")
default_value_flag = parameters.get("default_value_flag", True)

In [0]:
df = spark.read.table(source_view_name)

In [0]:
print("Source View Sanity Check")
print("Primary Keys Missing Check")

pk_null_expr = col(primary_keys[0]).isNull()
for clmn in primary_keys[1:]:
    pk_null_expr = pk_null_expr | col(clmn).isNull()

null_count = df.filter(pk_null_expr).count()

if null_count > 0:
    print(f"Found {null_count} records with null primary key(s):")
    display(df.filter(pk_null_expr).orderBy(*primary_keys))
    raise Exception("Null Primary Key(s) found.")
else:
    print("No null primary keys found.")

In [0]:
if default_value_flag:
    print("Default Value Mapping: ", default_value_flag)
    default_value_mapping = {
        "int": 0,
        "double": 0.0,
        "string": "N/A"
    }
    df = df.select(*[coalesce(col(clmn), lit(default_value_mapping.get(dtype, None))).alias(clmn) for clmn, dtype in df.dtypes])

In [0]:
enriched_df = df.withColumns({
    "creation_timestamp": lit(current_timestamp()),
    "updation_timestamp": lit(current_timestamp()),
    "job_run": lit(f"{job_id}{partition}"),
    "parent_job_run": lit(f"{parent_job}{partition}")
})

In [0]:
join_columns = primary_keys
update_columns = set(enriched_df.columns) - set(primary_keys) - set(['creation_timestamp'])

join_condition = " AND ".join([f"target.{clmn} = source.{clmn}" for clmn in join_columns]) + " AND " + \
    " AND ".join([f"target.{clmn} != source.{clmn}" for clmn in update_columns])

update_map = {f"{c}": f"source.{c}" for c in update_columns}

insert_columns = {f"{c}": f"source.{c}" for c in enriched_df.columns}

print("Join Conditions:", join_condition, "\n")
print("Update Columns:", update_map, "\n")
print("Insert Columns:", insert_columns, "\n")

In [0]:
enriched_df.filter("1 == 2").write.mode("overwrite").saveAsTable(target_table_name)

In [0]:
operation_metrics = DeltaTable.forName(spark, target_table_name).alias("target") \
    .merge(enriched_df.alias("source"), join_condition) \
    .whenMatchedUpdate(set=update_map) \
    .whenNotMatchedInsert(values=insert_columns) \
    .execute()
operation_metrics.display()

In [0]:
print("Running Post Merge sanity checks: ")
print("Duplicate Check")
source = enriched_df.alias("source")
target = spark.read.table(target_table_name).alias("target")

joined_df = source.join(
    target,
    on=[col(f"source.{clmn}") == col(f"target.{clmn}") for clmn in primary_keys],
    how="inner"
)

duplicate_df = joined_df.groupBy(*[col(f"source.{clmn}") for clmn in primary_keys]) \
    .agg(count("*").alias("row_count")) \
    .filter(col("row_count") > 1)

duplicate_count = duplicate_df.count()

if duplicate_count > 0:
    print(f"Found {duplicate_count} duplicate primary key(s):")
    display(duplicate_df.orderBy(col("row_count").desc()))
    raise Exception("Duplicate Records found.")
else:
    print("No duplicates found.")
